# 02 — Exploratory Data Analysis: Retinal Images

This notebook explores the retinal fundus image dataset.

**Goals:**
- Visualize sample fundus images per DR grade
- Test preprocessing (CLAHE, cropping, resizing)
- Verify augmentation pipeline
- Check image quality and size distributions

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from glob import glob

from src.config import load_settings
from src.data_prep.image_loader import (
    apply_clahe,
    resize_image,
    crop_to_circle,
    get_augmentation_pipeline,
    preprocess_single_image,
)

settings = load_settings()
plt.rcParams['figure.figsize'] = (14, 6)

print('Setup complete.')

## 1. Scan Image Directory

In [ ]:
image_dir = settings['paths']['raw_images']
extensions = ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff']
image_files = []
for ext in extensions:
    image_files.extend(glob(os.path.join(image_dir, ext)))

print(f'Found {len(image_files)} images in {image_dir}')

if len(image_files) > 0:
    print(f'Sample files: {[os.path.basename(f) for f in image_files[:5]]}')

## 2. Image Size Distribution

In [ ]:
widths, heights = [], []
for fpath in image_files[:500]:  # Sample up to 500 images
    img = cv2.imread(fpath)
    if img is not None:
        h, w = img.shape[:2]
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')

axes[1].hist(heights, bins=30, color='coral', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')

plt.tight_layout()
plt.show()

print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

## 3. Sample Images by DR Grade

In [ ]:
grade_labels = {0: 'No DR', 1: 'Mild NPDR', 2: 'Moderate NPDR', 3: 'Severe NPDR', 4: 'Proliferative DR'}

# Load label file
labels_path = os.path.join(os.path.dirname(settings['paths']['raw_tabular']), 'image_labels.csv')
if os.path.exists(labels_path):
    labels_df = pd.read_csv(labels_path)
    print(f'Labels loaded: {len(labels_df)} entries')
    print(labels_df['label'].value_counts().sort_index())
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for grade in range(5):
        subset = labels_df[labels_df['label'] == grade]
        if len(subset) > 0:
            fname = subset.iloc[0]['filename']
            fpath = os.path.join(image_dir, fname)
            if os.path.exists(fpath):
                img = cv2.imread(fpath)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[grade].imshow(img)
        axes[grade].set_title(f'Grade {grade}: {grade_labels[grade]}')
        axes[grade].axis('off')
    plt.suptitle('Sample Images per DR Grade', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f'Labels file not found at {labels_path}')

## 4. Preprocessing Pipeline Visualization

In [ ]:
if len(image_files) > 0:
    sample_path = image_files[0]
    raw = cv2.imread(sample_path)
    raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
    
    # Step-by-step preprocessing
    cropped = crop_to_circle(raw)
    enhanced = apply_clahe(cropped)
    resized = resize_image(enhanced)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    steps = [('Original', raw), ('Cropped', cropped), ('CLAHE Enhanced', enhanced), ('Resized (224x224)', resized)]
    for ax, (title, img) in zip(axes, steps):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.suptitle('Image Preprocessing Pipeline', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No images to process.')

## 5. Augmentation Preview

In [ ]:
if len(image_files) > 0:
    processed = preprocess_single_image(image_files[0])
    processed_uint8 = (processed * 255).astype(np.uint8)
    
    aug_pipeline = get_augmentation_pipeline(is_training=True)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes[0, 0].imshow(processed_uint8)
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    
    for i in range(1, 8):
        row, col = divmod(i, 4)
        augmented = aug_pipeline(image=processed_uint8)['image']
        axes[row, col].imshow(augmented)
        axes[row, col].set_title(f'Augmented #{i}')
        axes[row, col].axis('off')
    
    plt.suptitle('Data Augmentation Samples', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No images available for augmentation preview.')

## 6. Channel-wise Intensity Analysis

In [ ]:
if len(image_files) > 0:
    sample = preprocess_single_image(image_files[0])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    colors = ['red', 'green', 'blue']
    names = ['Red Channel', 'Green Channel', 'Blue Channel']
    
    for i, (color, name) in enumerate(zip(colors, names)):
        axes[i].hist(sample[:, :, i].ravel(), bins=50, color=color, alpha=0.7)
        axes[i].set_title(name)
        axes[i].set_xlabel('Pixel Intensity')
        axes[i].set_ylabel('Frequency')
    
    plt.suptitle('Channel-wise Pixel Intensity Distribution (after preprocessing)', fontsize=13)
    plt.tight_layout()
    plt.show()

## Summary

Key findings from image EDA:
1. **Image sizes** vary — resizing to 224x224 standardizes input
2. **CLAHE** significantly enhances vessel and lesion visibility
3. **Augmentation** pipeline provides good diversity without distorting clinical features
4. **Green channel** carries the most diagnostic information in fundus images